# 🚀 Manga Cleaner Backend trên Kaggle GPU + Cloudflare Tunnel

Notebook này chạy backend AI Manga Cleaner trực tiếp trên GPU chuyên dụng (Nvidia T4/P100).
- **Tốc độ:** ~0.5s - 1.2s / trang (nhanh gấp 10-20 lần so với CPU hoặc ZeroGPU bị nghẽn queue).
- **Kết nối:** Tự động tạo link HTTPS công khai qua **Cloudflare Tunnel** (`https://xxx.trycloudflare.com`) để frontend Vercel gọi thẳng.

In [ ]:
# [BƯỚC 1]: Kiểm tra GPU
!nvidia-smi

In [ ]:
# [BƯỚC 2]: Cài đặt thư viện cần thiết
!pip install -q fastapi uvicorn[standard] pydantic transformers>=4.50.0 opencv-python-headless pillow onnxruntime pyclipper httpx nest-asyncio

In [ ]:
# [BƯỚC 3]: Tải và cài đặt Cloudflare Tunnel (cloudflared)
!curl -L -s --output /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
!/tmp/cloudflared --version

In [ ]:
# [BƯỚC 4]: Tải trước các model inpainting và detector
import os, sys
from pathlib import Path

# Đảm bảo models directory tồn tại
os.makedirs("models", exist_ok=True)

!python download_models.py

In [ ]:
# [BƯỚC 5]: Khởi chạy Cloudflare Tunnel & Backend Server song song
import subprocess, time, re
from pathlib import Path

# 1. Chạy Cloudflare Tunnel ngầm
log_file = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# 2. Chờ 3-5 giây để lấy URL công khai
public_url = None
print("Đang kết nối Cloudflare Tunnel...")
for _ in range(25):
    time.sleep(1)
    if Path("/tmp/cloudflared.log").exists():
        content = Path("/tmp/cloudflared.log").read_text(errors="ignore")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

print("=" * 60)
if public_url:
    print("🎉 URL BACKEND CỦA BẠN ĐÃ SẴN SÀNG:")
    print(f"👉 {public_url}")
    print("=" * 60)
    print("Copy URL trên dán vào ô 'Cài đặt Backend' trên giao diện Web (Vercel) nhé!")
else:
    print("⚠️ Chưa bắt được URL tự động, kiểm tra lại log tại /tmp/cloudflared.log")
print("=" * 60)

# 3. Chạy Server FastAPI trên cổng 7860 (chạy foreground để giữ notebook luôn active)
!python server.py